In [1]:
#Testing, Validation & Business Insights
#Load Final Dataset
import pandas as pd

df = pd.read_csv('fraud_predictions.csv')

print(df.shape)
df.head()

(6362620, 10)


,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,transaction_frequency,velocity,isFraud,ai_fraud_flag,anomaly_score
0,244486.46,8946.00,0.00,526950.37,771436.84,1,999.0,0,0,0.411647
1,3170.28,58089.00,54918.72,0.00,0.00,1,999.0,0,0,0.433367
2,8424.74,783.00,0.00,0.00,0.00,1,999.0,0,0,0.439038
3,261877.19,7596.00,269473.19,1126627.70,864750.51,1,999.0,0,0,0.385170
4,20528.65,2302074.12,2322602.77,82696.17,62167.52,1,999.0,0,0,0.302614


In [2]:
#Check Missing Values
df.isnull().sum()


amount                   0
oldbalanceOrg            0
newbalanceOrig           0
oldbalanceDest           0
newbalanceDest           0
transaction_frequency    0
velocity                 0
isFraud                  0
ai_fraud_flag            0
anomaly_score            0
dtype: int64

In [3]:
#Check Fraud Distribution
df['isFraud'].value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [4]:
df['isFraud'].value_counts(normalize=True)*100

isFraud
0    99.870918
1     0.129082
Name: proportion, dtype: float64

In [5]:
#Validate AI Fraud Flags
df['ai_fraud_flag'].value_counts()

ai_fraud_flag
0    6354369
1       8251
Name: count, dtype: int64

In [6]:
#Create Confusion Matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    df['isFraud'],
    df['ai_fraud_flag']
)

print(cm)

[[6346215    8192]
 [   8154      59]]


In [7]:
#Calculate Fraud Capture Rate (FCR)
from sklearn.metrics import recall_score

fcr = recall_score(
    df['isFraud'],
    df['ai_fraud_flag']
)

print("Fraud Capture Rate:", fcr)

Fraud Capture Rate: 0.0071837331060513815


In [8]:
#Calculate False Positive Ratio (FPR)
tn, fp, fn, tp = cm.ravel()

fpr = fp / (fp + tn)

print("False Positive Ratio:", fpr)

False Positive Ratio: 0.0012891840261412277


In [9]:
#Calculate Value at Risk (VaR)
var = df[
    df['ai_fraud_flag'] == 1
]['amount'].sum()

print("Value at Risk:", var)

Value at Risk: 63283959900.759995


In [10]:
#Find Top 50 Suspicious Transactions
high_risk = df.sort_values(
    by='anomaly_score'
).head(50)

high_risk.head()

,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,transaction_frequency,velocity,isFraud,ai_fraud_flag,anomaly_score
4337015,30305608.13,0.00,0.00,32939427.28,63410883.66,2,67.0,0,1,-0.070323
4757443,38560.15,28210765.39,28249325.54,88456715.60,88418155.45,1,999.0,0,1,-0.069202
1698548,1379094.53,22268807.63,23647902.16,69906476.09,68527381.56,1,999.0,0,1,-0.069202
3088660,6392504.36,0.00,0.00,50536233.93,56928738.29,2,25.0,0,1,-0.068082
4373029,434940.33,24527239.23,24962179.57,57476124.97,57041184.64,1,999.0,0,1,-0.066964


In [11]:
high_risk.to_csv(
    'high_risk_transactions.csv',
    index=False
)